# multilingual-e5-large — Autoencoder + Agglomerative (Ward) clustering (deep clustering baseline)

The reference point for `clustering_e5_DCN.ipynb`, `clustering_e5_DEC.ipynb`, and
`clustering_e5_IDEC.ipynb`: train a plain autoencoder to compress each article\'s e5-large
embedding into a small latent space by minimizing **reconstruction error only** (no clustering
signal involved in training at all), then cluster that latent space with an ordinary shallow
algorithm — here, **Agglomerative (Ward-linkage) clustering**, swept over `distance_threshold` and
kept at whichever cut maximizes silhouette score, same as `clustering_e5_Agglomerative.ipynb`.

**Why Agglomerative and not K-Means as the shallow-clustering half of this baseline**: two papers
from the earlier literature review (arXiv:2511.19350, arXiv:2406.10552) found Agglomerative/HAC
the strongest of the "classical" (non-deep) clustering algorithms for short-text and news-event
clustering specifically — arXiv:2406.10552 in particular found it more robust than K-Means too, not
just HDBSCAN. Pairing the baseline with the best classical algorithm available (rather than an
arbitrary one) makes the comparison against DCN/DEC/IDEC fair: if the deep-clustering methods don\'t
beat *this*, the extra training complexity isn\'t earning its keep on this corpus.

This notebook is a pure control condition — the autoencoder here has no idea clustering is coming;
nothing in its training pushes the latent space to be cluster-friendly, only reconstruction-friendly.
DCN/DEC/IDEC all exist specifically to close that gap by folding the clustering objective into
training itself.

In [1]:
# Paths (Kaggle)
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

SEED = 42
np.random.seed(SEED)

DATA_DIR = Path("/kaggle/input/datasets/uom230425m/aggregation-pipeline-data/data")
RESULTS_DIR = Path("/kaggle/working/results/e5_ae_agglomerative")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data dir: {DATA_DIR.resolve()}")
print(f"Results dir: {RESULTS_DIR.resolve()}")

Data dir: /kaggle/input/datasets/uom230425m/aggregation-pipeline-data/data
Results dir: /kaggle/working/results/e5_ae_agglomerative


In [2]:
# Load annotator files
df_a = pd.read_csv(DATA_DIR / "a.csv")
df_d = pd.read_csv(DATA_DIR / "d.csv")
df_p = pd.read_csv(DATA_DIR / "p.csv")

for frame in (df_a, df_d, df_p):
    frame.drop(columns=["bias_label"], inplace=True, errors="ignore")

print(df_a.shape, df_d.shape, df_p.shape)

(750, 6) (750, 8) (800, 6)


In [3]:
# Concatenate dataframes and drop duplicate articles by article_id
df = pd.concat([df_a, df_d, df_p], ignore_index=True, sort=False)
df = df.drop_duplicates(subset="article_id", keep="first").reset_index(drop=True)
df.drop(columns=["flags", "Unnamed: 8"], inplace=True, errors="ignore")

print("df shape:", df.shape)
df.head()

df shape: (2000, 6)


,article_id,publisher,url,published_at,title,body_text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...


In [4]:
# Keep rows with the fields required for clustering
required_columns = ["article_id", "publisher", "url", "published_at", "title", "body_text"]
df = df.dropna(subset=required_columns).copy()

print("df shape after required-field dropna:", df.shape)
df.info()

df shape after required-field dropna: (1999, 6)
<class 'pandas.core.frame.DataFrame'>
Index: 1999 entries, 0 to 1999
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   article_id    1999 non-null   object
 1   publisher     1999 non-null   object
 2   url           1999 non-null   object
 3   published_at  1999 non-null   object
 4   title         1999 non-null   object
 5   body_text     1999 non-null   object
dtypes: object(6)
memory usage: 109.3+ KB


In [5]:
# Unicode normalization (NFC - canonical decomposition + composition)
for column in ["title", "body_text"]:
    df[column] = df[column].astype(str).map(lambda value: unicodedata.normalize("NFC", value))

df.head()

,article_id,publisher,url,published_at,title,body_text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...


In [6]:
# Remove title duplication from the beginning of the body text
def remove_title_from_body(row):
    body = row["body_text"].strip()
    title = row["title"].strip()
    if body.startswith(title):
        body = body[len(title):].lstrip("\n").lstrip()
    return body


df["text"] = df.apply(remove_title_from_body, axis=1)
df[["title", "text"]].head()

,title,text
0,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් හාල්මැස්සො නොව තෝරු මෝරු අහුවෙන කාලය බවමහ...
1,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',"මෙහි කිසිවක් අඩංගු නැත.Play video, ""දිරිය මිනි..."
2,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,සිගිරිය පිදුරංගල මාර්ගයේදී ඊයේ සවස වන අලියෙකු ...
3,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් දෙ...
4,ස්ථාන දෙකකදී ඝාතන දෙකක්,කුලී නිවසක පදිංචිව සිටි පුද්ගලයෙකුව ඊයේ (18) ර...


In [7]:
# Whitespace normalization
df["text"] = df["text"].map(lambda value: re.sub(r"\n{2,}", "\n", value))
df["text"] = df["text"].map(lambda value: re.sub(r"[ \t]+", " ", value))
df["text"] = df["text"].str.strip()

df = df[df["text"].str.len() > 0].reset_index(drop=True)
print("df shape after text cleaning:", df.shape)
df.head()

df shape after text cleaning: (1999, 7)


,article_id,publisher,url,published_at,title,body_text,text
0,f92797eb-a338-5886-a45a-66f01292a912,Lanka Deepa,https://www.lankadeepa.lk/news/දන-තර-මර-අහවන-ක...,2025-09-26 00:00:00+00:00,දැන් තෝරු-මෝරු අහුවෙන කාලේ,දැන් තෝරු-මෝරු අහුවෙන කාලේ\n\nදැන් හාල්මැස්සො ...,දැන් හාල්මැස්සො නොව තෝරු මෝරු අහුවෙන කාලය බවමහ...
1,5c7946aa-a810-59e2-a126-ff58347aad21,BBC Sinhala,https://www.bbc.com/sinhala/articles/c0vy04qd14yo,2024-01-17 00:00:00+00:00,අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනවා',අවුරුදු 67යි. ඇස් දෙකම පේන්නේ නෑ. මම පොල් කඩනව...,"මෙහි කිසිවක් අඩංගු නැත.Play video, ""දිරිය මිනි..."
2,6fc575d1-2be5-5d49-be20-c846617bdacd,Ada,https://www.ada.lk/breaking_news/පිදුරංගල-ගිය-...,2026-02-17 00:00:00+00:00,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට,පිදුරංගල ගිය විදේශිකයා වන අලි ප්‍රහාරයෙන් මරුට...,සිගිරිය පිදුරංගල මාර්ගයේදී ඊයේ සවස වන අලියෙකු ...
3,47965836-064f-530a-a34d-02f6898fb94a,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/194266,2024-03-07 00:00:00+00:00,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් ගැ...,හූති ප්‍රහාරයට ලක්වූ නෞකාවේ සිටි ලාංකිකයින් දෙ...
4,132990f7-a4f4-54ba-8b12-b469d9803bde,Ada Derana Sinhala,http://sinhala.adaderana.lk/news/195779,2024-04-19 00:00:00+00:00,ස්ථාන දෙකකදී ඝාතන දෙකක්,ස්ථාන දෙකකදී ඝාතන දෙකක්\n\nකුලී නිවසක පදිංචිව ...,කුලී නිවසක පදිංචිව සිටි පුද්ගලයෙකුව ඊයේ (18) ර...


In [8]:
# Build documents for embedding (title + body), with the "passage: " prefix multilingual-e5
# models require for corpus-side text
def build_passage_text(title, body):
    title = str(title).strip() if title else ""
    body = str(body).strip() if body else ""
    combined = f"{title}. {body}" if title else body
    return "passage: " + combined


df["passage_text"] = df.apply(lambda row: build_passage_text(row["title"], row["text"]), axis=1)
print(f"Documents: {len(df)}")
df["passage_text"].iloc[0][:500]

Documents: 1999


'passage: දැන් තෝරු-මෝරු අහුවෙන කාලේ. දැන් හාල්මැස්සො නොව තෝරු මෝරු අහුවෙන කාලය බවමහනුවර දිස්ත්\u200dරික් මන්ත්\u200dරී ජගත් මනුවර්ණ මහතා:-(ජා.ජ.බ) පාර්ලිමේන්තුවේදී පැවසීය.\nදූෂණයට විරුද්ධ වීම භයානක බවත් දූෂණයට විරුද්ධ නොවී සිටීම ඊට වඩා භයානක බවත් අනුර දිසානායක ජනාධිපති\xa0 එක්සත් ජාතීන්ගේ මහා මණ්ඩලයේ අමතමින් ප්\u200dරකාශ කළා.එය අප නැවත අවධාරණය කළ යුතුයි.අපි දේශපාලන පලි ගැනීම් කරනවා යැයි චෝදනා කරනවා.නමුත් ඇත්ත ඒක නෙමෙයි.මත්තල ගුවන් තොටුපොලේ මගින් පර්යන්තය එක\xa0 ආණ්ඩුවක් නෙළුම් පොහොට්ටුවක හැඩයකට හදන්න තීරණය කළාම ඊට පස'

In [9]:
# Embed with multilingual-e5-large (the finalized embedding model for this project),
# mean pooling — established as the best strategy for this model on this corpus
import torch
from transformers import AutoTokenizer, AutoModel

torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model_name = "intfloat/multilingual-e5-large"
tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)
model = AutoModel.from_pretrained(embedding_model_name).to(device)
model.eval()


def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


@torch.no_grad()
def embed_passages(texts, batch_size=16, max_length=512):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt",
        ).to(device)
        output = model(**encoded)
        pooled = mean_pooling(output.last_hidden_state, encoded["attention_mask"])
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        all_embeddings.append(pooled.cpu())
    return torch.cat(all_embeddings, dim=0).numpy()


embeddings = embed_passages(df["passage_text"].tolist(), batch_size=16)
print(embeddings.shape)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


OutOfMemoryError: CUDA out of memory. Tried to allocate 978.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 536.75 MiB is free. Process 105 has 2.79 GiB memory in use. Process 174 has 2.79 GiB memory in use. Process 229 has 2.79 GiB memory in use. Process 281 has 2.79 GiB memory in use. Process 342 has 2.79 GiB memory in use. Including non-PyTorch memory, this process has 102.00 MiB memory in use. Of the allocated memory 0 bytes is allocated by PyTorch, and 0 bytes is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Standardize before feeding the autoencoder (zero mean / unit variance per dimension) —
# standard practice for deep clustering methods, keeps every input dimension on a comparable
# scale for the reconstruction loss even though the raw embeddings are already L2-normalized
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(embeddings).astype(np.float32)
X_tensor = torch.tensor(X, dtype=torch.float32)
print(X.shape)

In [ ]:
# Shared autoencoder architecture used by every notebook in this folder
import torch.nn as nn

INPUT_DIM = X.shape[1]
HIDDEN_DIMS = [256, 64]
LATENT_DIM = 16


class Autoencoder(nn.Module):
    def __init__(self, input_dim=INPUT_DIM, hidden_dims=HIDDEN_DIMS, latent_dim=LATENT_DIM):
        super().__init__()
        enc_layers, prev = [], input_dim
        for h in hidden_dims:
            enc_layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        enc_layers += [nn.Linear(prev, latent_dim)]
        self.encoder = nn.Sequential(*enc_layers)

        dec_layers, prev = [], latent_dim
        for h in reversed(hidden_dims):
            dec_layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        dec_layers += [nn.Linear(prev, input_dim)]
        self.decoder = nn.Sequential(*dec_layers)

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return z, x_hat

In [ ]:
# Pretrain the autoencoder on reconstruction loss only
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(SEED)
autoencoder = Autoencoder().to(device)
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)
loader = DataLoader(TensorDataset(X_tensor), batch_size=256, shuffle=True)

N_EPOCHS = 100
for epoch in range(N_EPOCHS):
    total_loss = 0.0
    for (batch,) in loader:
        batch = batch.to(device)
        z, x_hat = autoencoder(batch)
        loss = nn.functional.mse_loss(x_hat, batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(batch)
    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"epoch {epoch + 1:>3}/{N_EPOCHS}  reconstruction MSE: {total_loss / len(X_tensor):.4f}")

In [ ]:
# Extract the latent representation for every document
autoencoder.eval()
with torch.no_grad():
    final_latent, _ = autoencoder(X_tensor.to(device))
    final_latent = final_latent.cpu().numpy()
print(final_latent.shape)

In [ ]:
# Sweep Agglomerative (Ward) distance_threshold on the latent space, pick the cut that
# maximizes silhouette score.
#
# BUG FIXED: this originally reused clustering_e5_Agglomerative.ipynb\'s fixed threshold range
# (0.5-15.0), which was tuned for that notebook\'s 5-D UMAP/cosine-metric space, where distances
# stay small by construction. This autoencoder\'s 16-D latent space has no such bound (no
# normalization or bounded activation on the bottleneck layer) — empirically its pairwise
# distances run from the low teens to the 60s, so a 0.5-15.0 sweep barely reached valid territory:
# it silently produced near-zero merging (thousands of clusters, silhouette ~0) instead of
# crashing, which is worse than an error because it looks like it ran successfully. Fixed by
# deriving the sweep range from this latent space\'s own observed pairwise-distance distribution
# instead of a hardcoded range.
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.spatial.distance import pdist

rng_scale = np.random.default_rng(SEED)
distance_sample = final_latent[rng_scale.choice(len(final_latent), size=min(300, len(final_latent)), replace=False)]
pairwise_distances = pdist(distance_sample)
threshold_low, threshold_high = np.percentile(pairwise_distances, [2, 80])
candidate_thresholds = np.linspace(threshold_low, threshold_high, 30)
print(f"latent pairwise distances observed: [{pairwise_distances.min():.2f}, {pairwise_distances.max():.2f}]")
print(f"sweeping distance_threshold over: [{threshold_low:.2f}, {threshold_high:.2f}]")

sweep_results = []
for threshold in candidate_thresholds:
    model = AgglomerativeClustering(n_clusters=None, distance_threshold=threshold, linkage="ward")
    labels = model.fit_predict(final_latent)
    n_found = len(set(labels))
    if 1 < n_found < len(labels):
        sil = silhouette_score(final_latent, labels, metric="cosine")
        sweep_results.append((threshold, n_found, sil))

if not sweep_results:
    raise RuntimeError(
        "No distance_threshold in the swept range produced a valid clustering (>1 and <n clusters). "
        "Try widening the percentile range in threshold_low/threshold_high above."
    )

sweep_df = pd.DataFrame(sweep_results, columns=["distance_threshold", "n_clusters", "silhouette"])
sweep_df = sweep_df.sort_values("silhouette", ascending=False)
sweep_df.head(10)

In [ ]:
# Refit at the best threshold found
best_threshold = float(sweep_df.iloc[0]["distance_threshold"])
print(f"Best distance_threshold: {best_threshold:.3f} (n_clusters={int(sweep_df.iloc[0]['n_clusters'])})")

model = AgglomerativeClustering(n_clusters=None, distance_threshold=best_threshold, linkage="ward")
final_labels = model.fit_predict(final_latent)

print(pd.Series(final_labels).value_counts().head(20))
print("Number of clusters found:", len(set(final_labels)))

In [ ]:
# Clustering evaluation (cosine metric, matching every other notebook in this project)
from sklearn.metrics import davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics.pairwise import cosine_similarity

n_clusters = len(set(final_labels))
sil = silhouette_score(final_latent, final_labels, metric="cosine")
dbi = davies_bouldin_score(final_latent, final_labels)
ch = calinski_harabasz_score(final_latent, final_labels)

print(f"Model: {embedding_model_name} + Autoencoder + Agglomerative (Ward)")
print(f"Articles: {len(df)} | Clusters: {n_clusters} | Noise ratio: 0.00% (n/a for this algorithm)")
print(f"Silhouette Score (cosine): {sil:.4f}")
print(f"Davies-Bouldin Index:      {dbi:.4f}  (lower is better)")
print(f"Calinski-Harabasz Index:   {ch:.2f}  (higher is better)")

# Separation score computed on the raw e5 embeddings, for comparability with every other
# notebook in this project (not the AE latent space, which is method-specific)
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(embeddings), size=min(100, len(embeddings)), replace=False)
sims = cosine_similarity(embeddings[sample_idx])
pairwise = sims[np.triu_indices_from(sims, k=1)]
separation_score = 1 - pairwise.mean()

scores_path = RESULTS_DIR / "e5_ae_agglomerative_scores.csv"
row = {
    "model": embedding_model_name,
    "pipeline": "Autoencoder+Agglomerative(Ward)",
    "embedding_dim": embeddings.shape[1],
    "latent_dim": LATENT_DIM,
    "separation_score": round(separation_score, 4),
    "n_articles": len(df),
    "n_clusters": n_clusters,
    "noise_ratio": 0.0,
    "silhouette": round(sil, 4),
    "davies_bouldin": round(dbi, 4),
    "calinski_harabasz": round(ch, 2),
    "distance_threshold": round(best_threshold, 3),
}
pd.DataFrame([row]).to_csv(scores_path, index=False)
print(f"Saved scores to {scores_path}")

In [ ]:
# Inspect sample titles per cluster
df["cluster_id"] = final_labels
for cluster_id, group in list(df.groupby("cluster_id"))[:10]:
    print(f"=== Cluster {cluster_id} ({len(group)} articles) ===")
    for title in group["title"].head(10):
        print(f"- {title}")
    print()

In [ ]:
# Inspect a RANDOM sample of clusters (rather than just the first few by id) — a more
# representative check of overall cluster quality than always looking at the same low-numbered
# clusters
rng_inspect = np.random.default_rng(SEED)
cluster_ids = df.loc[df["cluster_id"] != -1, "cluster_id"].unique()
sample_size = min(8, len(cluster_ids))
sampled_cluster_ids = rng_inspect.choice(cluster_ids, size=sample_size, replace=False)

for cluster_id in sampled_cluster_ids:
    group = df[df["cluster_id"] == cluster_id]
    print(f"=== Cluster {cluster_id} ({len(group)} articles) ===")
    for title in group["title"].head(15):
        print(f"- {title}")
    print()

In [ ]:
# Save article-level assignments
assignments_path = RESULTS_DIR / "e5_ae_agglomerative_assignments.csv"
df.drop(columns=["passage_text"], errors="ignore").to_csv(assignments_path, index=False, encoding="utf-8-sig")
print(f"Saved assignments: {assignments_path.resolve()}")